# 2.2 Refactorización de P2

Lo elegido en `1.2-dm-p2-exploracion-riesgo` pasa a clases con `fit` / `score`, una por modelo, y a funciones para umbral y métricas. Al final se compara con `mibici_equipo39/modeling/p2.py`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import TargetEncoder

from mibici_equipo39 import config as C
from mibici_equipo39.features import FEATURE_COLUMNS, load_p2_targets, load_tables

metadata, X, _ = load_tables()
targets = load_p2_targets()

## Modelos como clases

Las dos clases reciben el mismo `frame` (metadata + X) y devuelven un puntaje por fila, así el experimento no necesita saber qué modelo está usando.

In [2]:
class SeasonalRateBaseline:
    levels = (["station_id", "is_weekend", "hour_of_day"], ["station_id"])

    def fit(self, frame, y):
        self.rates_ = [y.groupby([frame[k] for k in keys], observed=True).mean().rename("rate") for keys in self.levels]
        self.global_ = float(y.mean())
        return self

    def score(self, frame):
        out = pd.Series(np.nan, index=frame.index)
        for keys, rates in zip(self.levels, self.rates_):
            out = out.fillna(frame[keys].join(rates, on=keys)["rate"])
        return out.fillna(self.global_).to_numpy()


ENCODED = ["te_station", "te_station_hour"]

class TargetEncodedHGB:
    def _keys(self, frame):
        station = frame["station_id"].astype(str)
        return pd.DataFrame({"te_station": station,
                             "te_station_hour": station + "|" + frame["is_weekend"].astype(str) + "|" + frame["hour_of_day"].astype(str)})

    def fit(self, frame, y):
        self.encoder_ = TargetEncoder(target_type="binary", cv=5, shuffle=True, random_state=C.SEED)
        encoded = pd.DataFrame(self.encoder_.fit_transform(self._keys(frame), y), columns=ENCODED, index=frame.index)
        self.model_ = HistGradientBoostingClassifier(**C.P2_HGB_PARAMS).fit(pd.concat([frame[FEATURE_COLUMNS], encoded], axis=1), y)
        return self

    def score(self, frame):
        encoded = pd.DataFrame(self.encoder_.transform(self._keys(frame)), columns=ENCODED, index=frame.index)
        return self.model_.predict_proba(pd.concat([frame[FEATURE_COLUMNS], encoded], axis=1))[:, 1]

> **Por qué `fit` hace el encoding.** Si el `TargetEncoder` se ajustara fuera del modelo, sería fácil aplicarlo con datos de validación por error. Dentro de `fit` solo ve train, con validación cruzada interna, y `score` solo transforma.

## Umbral y métricas

In [3]:
def choose_threshold(y, score, min_recall=C.P2_MIN_RECALL):
    precision, recall, thresholds = precision_recall_curve(y, score)
    table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
    table["f1"] = (2 * table["precision"] * table["recall"] / (table["precision"] + table["recall"])).fillna(0)
    ok = table.loc[table["recall"].ge(min_recall)]
    pool, column = (ok, "precision") if len(ok) else (table, "f1")
    return float(pool.loc[pool[column].eq(pool[column].max()), "threshold"].max())


def classification_metrics(y, score, threshold):
    alert = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, alert, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"recall": recall, "precision": precision, "f1": f1, "ap": average_precision_score(y, score), "alert_rate": float(alert.mean())}

> **Si no se alcanza el recall mínimo** se cae a mayor F1 en lugar de fallar. En empate se queda el umbral más alto, que genera menos visitas.

## Experimento

In [4]:
frame = pd.concat([metadata, X, targets], axis=1)
frame = frame.loc[frame["capacity"].notna()].reset_index(drop=True)
train = frame.loc[frame["split"].eq("train")]
val = frame["split"].eq("validation").to_numpy()

notebook_scores, rows = {}, {}
for kind, model in [("baseline", SeasonalRateBaseline), ("candidato", TargetEncodedHGB)]:
    row = {}
    for risk, target in [("bikes", "y_bikes"), ("docks", "y_docks")]:
        y = frame[target].astype(int)
        score = model().fit(train, y[train.index]).score(frame)
        notebook_scores[(kind, risk)] = score
        t = choose_threshold(y.to_numpy()[val], score[val])
        row.update({f"val_{k}_{risk}": v for k, v in classification_metrics(y.to_numpy()[val], score[val], t).items()})
    row["val_macro_f1"] = (row["val_f1_bikes"] + row["val_f1_docks"]) / 2
    rows[kind] = row
pd.DataFrame(rows).T[["val_recall_bikes", "val_precision_bikes", "val_recall_docks", "val_precision_docks", "val_macro_f1"]].round(4)

,val_recall_bikes,val_precision_bikes,val_recall_docks,val_precision_docks,val_macro_f1
baseline,0.8188,0.0482,0.8213,0.0850,0.1225
candidato,0.8003,0.0589,0.8001,0.0909,0.1365


> **Mismos números que la exploración.** Prueba se calcula solo dentro del run de MLflow.

## Paso al módulo

| Pieza | Destino |
|---|---|
| Etiquetas `proxy_labels`, `make_capacity_proxy_targets` | `mibici_equipo39/features.py` |
| `SeasonalRateBaseline`, `TargetEncodedHGB`, `choose_threshold`, `classification_metrics`, `P2Experiment` | `mibici_equipo39/modeling/p2.py` |
| Ocupación, margen, recall mínimo e hiperparámetros | `config.py` |
| Curva precisión-recall | `mibici_equipo39/plots.py` |

Se ejecuta con `python -m mibici_equipo39.modeling.p2 --executor personaN` o con `mlflow run . -e p2`.

In [5]:
from mibici_equipo39.modeling.p2 import P2Experiment

module = P2Experiment(metadata, X, targets).fit()
for kind in ["baseline", "candidato"]:
    for risk in ["bikes", "docks"]:
        assert np.allclose(notebook_scores[(kind, risk)], module.scores[kind][risk])
print("notebook y módulo dan los mismos puntajes; seleccionado:", module.selected)

notebook y módulo dan los mismos puntajes; seleccionado: candidato
